In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='2'
import sys
import random

import numpy as np
import torch
from torch.cuda.memory import set_per_process_memory_fraction
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import OneCycleLR

from gen_bert_embedding import circRNABert, seq2kmer_bert

import torch.utils.data
from transformers import BertModel, BertTokenizer
from utils import read_csv, convert_one_hot2

from MyNet1 import *
from train_loop import train, validate
from utils import read_h5, myDataset, GradualWarmupScheduler, param_num, split_dataset, convert_one_hot

def seq2kmer_bert(seq, k):
    seq_length = len(seq)
    import random
    kmer = [seq[x:x + k] for x in range(seq_length - k + 1)]
    kmers = " ".join(kmer)
    return kmers

In [2]:
max_length = 101

# filename = sys.argv[1] + '.tsv'
# TARDBP_K562

# file_name = 
# filename = 'TIA1_Hela.tsv'
'''
filename = 'LIN28B_HepG2.tsv'
base_path = '/home/zhuhaoran/MyNet/'

sequences, structs, label = read_csv(base_path + 'clip_data/' + filename)
'''


filename = 'LIN28B_HepG2.tsv'
filename2 = 'LIN28B_K562.tsv'

base_path = '/home/zhuhaoran/MyNet/'

sequences, structs, label = read_csv(base_path + 'clip_data/' + filename)

sequences2, structs2, label2 = read_csv(base_path + 'clip_data/' + filename2)

sequences = list(sequences)
sequences2 = list(sequences2)

label = list(label.squeeze())
label2 = list(label2.squeeze())
to_plot = []
for seq, lb1 in zip(sequences, label):
    for seq2, lb2 in zip(sequences2, label2):
        if seq == seq2:
            if lb1!=lb2:
                print(seq)

to_plot.append(seq)

structure = np.zeros((len(structs), 1, max_length))  # (N, 1, 101)
for i in range(len(structs)):
    struct = structs[i].split(',')
    ti = [float(t) for t in struct]
    ti = np.array(ti).reshape(1, -1)
    structure[i] = np.concatenate([ti], axis=0)

structure2 = np.zeros((len(structs2), 1, max_length))  # (N, 1, 101)
for i in range(len(structs2)):
    struct = structs2[i].split(',')
    ti = [float(t) for t in struct]
    ti = np.array(ti).reshape(1, -1)
    structure2[i] = np.concatenate([ti], axis=0)


# 生成bert embedding
model__path = '/home/zhuhaoran/PrismNet-master/3-new-12w-0/'  
tokenizer = BertTokenizer.from_pretrained(model__path, do_lower_case=False)
model = BertModel.from_pretrained(model__path)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
# model = torch.nn.DataParallel(model, device_ids=[0, 1, 2, 3])
model = model.eval()


bert_embedding = circRNABert(to_plot, model, tokenizer, device, 3)  # default k=3  # (N, 101, 768)
bert_embedding = bert_embedding.transpose([0, 2, 1])  # (N, 768, 101)

Some weights of the model checkpoint at /home/zhuhaoran/PrismNet-master/3-new-12w-0/ were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MyNet10().to(device)

# modelname = filename.replace('K562', 'HepG2')
# modelname = filename.replace('HepG2', 'K562')
model_file = '/home/zhuhaoran/MyNet/out/out1/' + filename + '_best.pth'
model.load_state_dict(torch.load(model_file))

<All keys matched successfully>

In [5]:
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(2))

model.eval()
output = model(x, s)
loss = criterion(output, y)
prob = torch.sigmoid(output)


In [23]:
# outputs = torch.tensor(yyy)

ind = np.where(p_all>=0.999)[0]
l = len(ind)

import random
 
i = random.randint(0,l)
i = ind[i]

outputs = torch.tensor(p_all)
print(outputs[i],
test_label[i],
i,

seq2kmer_bert(test_seq[i], 3))

tensor([0.9995]) [1.] 258 ATC TCT CTT TTC TCC CCC CCA CAC ACT CTC TCC CCA CAG AGA GAT ATC TCC CCC CCA CAA AAA AAG AGA GAG AGA GAA AAG AGA GAT ATT TTG TGA GAT ATA TAT ATT TTG TGG GGG GGC GCT CTG TGC GCC CCT CTC TCC CCA CAC ACC CCC CCA CAA AAA AAG AGT GTC TCT CTC TCC CCC CCG CGG GGA GAC ACC CCC CCA CAA AAC ACA CAG AGC GCT CTA TAC ACT CTG TGG GGA GAA AAC ACG CGG GGA GAA AAA AAC ACA CAG AGG GGC GCC CCA CAT ATC TCC CCA CAG


In [24]:
import shap
import pandas as pd
torch.cuda.empty_cache()
test_emb = torch.tensor(test_emb).requires_grad_().to(device).type(torch.float32)
test_struc = torch.tensor(test_struc).requires_grad_().to(device).type(torch.float32)
# print(test_emb)
e = shap.GradientExplainer(model, [test_emb, test_struc])

shap_values = e.shap_values([test_emb[i:i+1], test_struc[i:i+1]])  # 选择最显著的那个

def norm(data):
    _range = np.max(data,axis=2) - np.min(data, axis=2)
    return (data - np.min(data,axis=2).transpose(1,0)) / _range.transpose(1,0)

sss = seq2kmer_bert(test_seq[i], 3).split(' ')

input_seq = test_seq[i]
input_struct = test_struc[i]

# bert_gradient_data = norm(shap_values[0]) #[1, 768, 99]
# bert_gradient_data = np.exp(shap_values[0])

bert_attention_data = np.max(shap_values[0], axis=1)*10000  # [1,99]  # 求了平均 
bert_attention_data = np.insert(bert_attention_data, 0, values=0, axis=1)
bert_attention_data = np.insert(bert_attention_data, bert_attention_data.shape[-1], values=0, axis=1)  # [1,1,101], 左右两边补了俩个0
from utils import convert_one_hot2
bert_attention_data = convert_one_hot2([input_seq], bert_attention_data[0,:]).transpose(1,0,2).squeeze()

struc_attention_data = shap_values[1][0]*10000
# (shap_values[1][0]-np.min(shap_values[1][0]))/(np.max(shap_values[1][0])-np.min(shap_values[1][0]))

W = np.concatenate([bert_attention_data, struc_attention_data], axis=0)

x_str = input_struct.cpu().detach().numpy().reshape(101, 1)
str_null = np.zeros_like(x_str)
ind =np.where(x_str == -1)[0]
str_null[ind,0]=1
str_null=np.squeeze(str_null).T


onehot = convert_one_hot([input_seq]).squeeze()

X = np.concatenate([onehot, input_struct.cpu().detach().numpy()], axis=0)



# df = pd.DataFrame(data) #, columns=sss)

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).


torch.Size([3000, 768, 99]) torch.Size([3000, 1, 101])
[0, 8, 13, 15, 20, 21, 22, 24, 26, 27, 29, 33, 35, 49, 53, 54, 55, 65, 69, 70, 72, 76, 81, 82, 86, 87, 88, 90, 95, 99]
[0, 8, 13, 15, 20, 21, 22, 24, 26, 27, 29, 33, 35, 49, 53, 54, 55, 65, 69, 70, 72, 76, 81, 82, 86, 87, 88, 90, 95, 99]


In [25]:
#from skimage.transform import resize
import importlib
import visualize
importlib.reload(visualize)
visualize.plot_saliency(X, W, nt_width=100, norm_factor=3, str_null=str_null, outdir="./dynamic_motif/out.pdf")

In [9]:
y = np.mean(shap_values[0], axis=1)

CTTCCTTTTTTCTTTTTTCCGGCGTTCAAGATGTCGAAGCGAGGACGTGGTGGGTCCTCTGGTGCGAAATTCCGGATTTCCTTGGGTCTTCCGGTAGGAGC
AAGACTTCCCTCAGATGGGTCGTTTCACCTTAAGAGATGAGGGTAAGACCATTGCAATTGGAAAAGTTCTGAAACTGGTTCCAGAGAAAGACTAAGCATTT
AAGCAAAAATCTGCCTTGAGATCATGCAGAGAACTGGTGCTCACTTGGAGCTGTCTTTGGCCAAAGACCAAGGCCTCTCCATCATGGTGTCAGGAAAGCTG
CCGCCGCTCCGCCGTCGCGTTTCTCTGCCGGTCGCAATGGAAGAAGAGATCGCCGCGCTGGTCATTGACAATGGCTCCGGCATGTGCAAAGCTGGTTTTGC
CCACGTCCTGCAGCTGCAGCCGCTGCAGCTACTCCTGCTGTCCGCACCGTTCCACAGTATAAATATGCTGCAGGAGTTCGCAATCCTCAGCAACATCTTAA
CGCGTTTCTCTGCCGGTCGCAATGGAAGAAGAGATCGCCGCGCTGGTCATTGACAATGGCTCCGGCATGTGCAAAGCTGGTTTTGCTGGGGACGACGCTCC


In [8]:
list(label)

[array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], dtype=float32),
 array([1.], d